# Stage 3 — SARIMA Seasonal Baseline

SARIMAX with Fourier seasonal terms, fit per-station on 4 stations representative of the 4 EDA regions, then pooled.

## 1. Imports

In [2]:
import sys
!{sys.executable} -m pip install statsmodels

import pandas as pd
import numpy as np
import json, time, warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 9.2 MB/s eta 0:00:02
   -- ------------------------------------- 0.8/11.3 MB 1.9 MB/s eta 0:00:06
   -- ------------------------------------- 0.8/11.3 MB 1.9 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/11.3 MB 1.4 MB/s eta 0:00:08
   ---- ----------------------------------- 1.3/11.3 MB 1.4 MB/s eta 0:00:08
   ------ --------------------------------- 1.8/11.3 MB 1.3 MB/s eta 0:00:08
   ------- -------------------------------- 2.1/11.3 MB 1.3 MB/s eta 0:00:08
   -------- ------------------------------- 2.4/11.3 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.6/11.3 MB 1.4 MB/s eta 0:00:07
   ------------ --------------------------- 3.7/11.3 MB 1.7 MB/s eta 0:00:05
   ----------------- ---------------------- 5.0/11.3 MB 2.1 MB/s eta 0:00:03
   ---------------------- ----------------- 6.3/11.3 MB 2.5 MB/s eta 0:00:03
   ---

## 2. Load data & build the next-day rainfall target

In [3]:
t0 = time.time()
df = pd.read_csv("../Data/Processed/Processed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["Station", "date"]).reset_index(drop=True)
df["Target_Rainfall"] = df.groupby("Station")["BMD_Rainfall"].shift(-1)
df["next_date"] = df.groupby("Station")["date"].shift(-1)
gap_days = (df["next_date"] - df["date"]).dt.days
valid = df["Target_Rainfall"].notna() & (gap_days == 1)
df = df[valid].drop(columns=["next_date"]).reset_index(drop=True)

## 3. Fourier seasonal terms

4 representative stations, one per EDA region. `K=3` harmonics approximate annual seasonality on daily data (period ≈ 365.25 days).

In [4]:
# 4 representative stations covering the 4 EDA regions
stations = ["Sylhet", "Dhaka", "Rajshahi", "Barisal"]
K = 3  # Fourier harmonics for annual seasonality (daily data, period=365.25)
period = 365.25

def fourier_terms(doy, K, period):
    t = 2 * np.pi * doy / period
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(k * t)
        cols[f"cos{k}"] = np.cos(k * t)
    return pd.DataFrame(cols)

## 4. Fit SARIMAX per station & forecast one-step-ahead

In [5]:
all_metrics = {}
per_station_preds = []

for st in stations:
    sub = df[df["Station"] == st].sort_values("date").reset_index(drop=True)
    exog = fourier_terms(sub["DOY"].values, K, period)
    exog.index = sub.index
    y = sub["Target_Rainfall"]

    train_mask = sub["Year"] <= 2011  # train+val, matching the ML models' train+val window
    test_mask = sub["Year"] >= 2012

    y_train, exog_train = y[train_mask], exog[train_mask]
    y_test, exog_test = y[test_mask], exog[test_mask]

    st0 = time.time()
    mod = SARIMAX(y_train, exog=exog_train, order=(1, 0, 1),
                   enforce_stationarity=False, enforce_invertibility=False)
    res = mod.fit(disp=False, maxiter=50)

    # one-step-ahead forecasts across the test period using true history (matches
    # how the ML models' lag features also use true past values, not model-predicted ones)
    extended = res.append(y_test, exog=exog_test, refit=False)
    pred = extended.get_prediction(start=len(y_train), end=len(y_train) + len(y_test) - 1,
                                    dynamic=False).predicted_mean.values
    pred = np.clip(pred, 0, None)  # rainfall can't be negative

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = float(mean_absolute_error(y_test, pred))
    r2 = float(r2_score(y_test, pred))
    all_metrics[st] = {"RMSE": rmse, "MAE": mae, "R2": r2, "n_test": int(test_mask.sum())}
    per_station_preds.append((y_test.values, pred))
    print(f"{st:10s} order=(1,0,1) Fourier(K={K})  RMSE={rmse:6.3f} MAE={mae:6.3f} R2={r2:7.4f}  ({time.time()-st0:.1f}s)")

Sylhet     order=(1,0,1) Fourier(K=3)  RMSE=22.476 MAE=10.440 R2= 0.1637  (3.4s)
Dhaka      order=(1,0,1) Fourier(K=3)  RMSE=10.907 MAE= 4.643 R2= 0.1178  (2.9s)
Rajshahi   order=(1,0,1) Fourier(K=3)  RMSE=10.283 MAE= 4.050 R2= 0.0511  (2.7s)
Barisal    order=(1,0,1) Fourier(K=3)  RMSE=14.828 MAE= 5.745 R2= 0.1508  (3.8s)


## 5. Pooled metrics across the 4 stations & save

In [6]:
y_pool = np.concatenate([p[0] for p in per_station_preds])
pred_pool = np.concatenate([p[1] for p in per_station_preds])
rmse = float(np.sqrt(mean_squared_error(y_pool, pred_pool)))
mae = float(mean_absolute_error(y_pool, pred_pool))
r2 = float(r2_score(y_pool, pred_pool))
all_metrics["POOLED_4_STATIONS"] = {"RMSE": rmse, "MAE": mae, "R2": r2, "n_test": int(len(y_pool))}
print(f"POOLED (4 stations)  RMSE={rmse:.3f} MAE={mae:.3f} R2={r2:.4f}")

os.makedirs("../outputs", exist_ok=True)
pd.DataFrame(all_metrics).T.to_csv("../outputs/model-results/stage3_sarima_seasonal_baseline_results.csv")
print("done sarima", time.time() - t0)

POOLED (4 stations)  RMSE=15.409 MAE=6.220 R2=0.1717
done sarima 14.751709222793579
